## Тест
Важно, чтобы сервер был поднят на astro_env и тянул оттуда все импорты

In [7]:
import sys
print(sys.executable)

[22:04:50] /Users/zuha/Desktop/FKI/4 курс 2025-2026/course_work-SBF/astro_env/bin/python3


In [49]:
import argparse, numpy as np
import time
import builtins
from pathlib import Path
from astropy.io import fits
from astropy.wcs import WCS
from astropy.stats import sigma_clipped_stats
from scipy.ndimage import gaussian_filter
from scipy.signal import fftconvolve
from astropy.convolution import Gaussian2DKernel, interpolate_replace_nans
from photutils.isophote import Ellipse, EllipseGeometry, build_ellipse_model
from photutils.segmentation import detect_sources, deblend_sources
import stpsf                                    # PSF под JWST файл
from scipy.fft import fft2, fftfreq, fftshift, set_workers
from photutils.background import Background2D, MedianBackground
from astropy.stats import SigmaClip
from astropy.modeling.models import Sersic2D
from astropy.modeling.fitting import LevMarLSQFitter
# Все print в файле автоматически получают префикс с текущим временем устройства
_orig_print = builtins.print

def print(*args, **kwargs):
    _orig_print(f"[{time.strftime('%H:%M:%S')}]", *args, **kwargs)



In [22]:
# --- жёстко заданные входные файлы ---
f150w_path = Path("../data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d.fits")
f090w_path = Path("../data/NGC 1380/jw03055-o001_t001_nircam_clear-f090w_i2d.fits")

# --- куда писать результаты ---
out_dir = f150w_path.parent
stem = f150w_path.stem

# --- режимы ---
USE_ISOPHOTE = True
DUMP_ISOPHOTES = True
USE_BKG = False          # для начала лучше False, раз фон пока только мешал
DO_DEBLEND = True       # для начала лучше True, иначе SBF будет сильно испорчен яркими источниками
DRY = False

# --- центр ---
FIXED_CENTER = None      # пример: (6306.0, 2730.0)
USE_FAST_CENTER = True   # если True, то центр будет найден быстро (но менее точно) на разреженной сетке, иначе - на всех пикселях

# --- background/mask ---
BKG_BOX0 = 256
BKG_BOX2D = 256
MASK_NSIGMA = 2.5
MASK_NPIXELS = 5

# --- isophote ---
HALF_SIZE = 3000
ISO_START_SMA = 50.0
ISO_START_EPS = 0.2
ISO_START_PA = 0.0

# --- FFT / PSF ---
FFT_WORKERS = -1         # -1 = все доступные потоки scipy.fft
PSF_SIZE = 129
PSF_NLAMBDA = 7
PSFREF = None            # если None, считаем PSF по f150w_path

# --- SBF annulus ---
RIN_SBF = 50.0
ROUT_SBF = 200.0

Это бывший load_i2d, но в развёрнутом виде.

In [3]:
print("loading i2d files...")

# F150W
with fits.open(f150w_path, memmap=False) as hdul:
    img_f150 = hdul["SCI"].data.astype(float)
    hdr150 = hdul["SCI"].header
    valid150 = np.isfinite(img_f150)

    if "WHT" in hdul:
        try:
            wht150 = hdul["WHT"].data
            valid150 &= np.isfinite(wht150) & (wht150 > 0)
        except Exception:
            pass

wcs150 = WCS(hdr150)
pixar_sr = float(hdr150["PIXAR_SR"])
pix_area = pixar_sr / 2.350443e-11
print(f"[WCS] PIXAR_SR={pixar_sr:.8e} sr → pix_area={pix_area:.8e} arcsec^2")

# F090W (опционально)
img_f090 = None
hdr090 = None
valid090 = None
wcs090 = None

if f090w_path.exists():
    with fits.open(f090w_path, memmap=False) as hdul:
        img_f090 = hdul["SCI"].data.astype(float)
        hdr090 = hdul["SCI"].header
        valid090 = np.isfinite(img_f090)

        if "WHT" in hdul:
            try:
                wht090 = hdul["WHT"].data
                valid090 &= np.isfinite(wht090) & (wht090 > 0)
            except Exception:
                pass

    wcs090 = WCS(hdr090)

print(f"F150W shape = {img_f150.shape}, valid = {valid150.sum()}")
if img_f090 is not None:
    print(f"F090W shape = {img_f090.shape}, valid = {valid090.sum()}")

[21:38:12] loading i2d files...


Set DATE-AVG to '2023-11-09T21:33:24.076' from MJD-AVG.
Set DATE-END to '2023-11-09T21:52:00.860' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to    23.109100 from OBSGEO-[XYZ].
Set OBSGEO-H to 1523626142.965 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


[21:38:12] [WCS] PIXAR_SR=2.29190411e-14 sr → pix_area=9.75094527e-04 arcsec^2
[21:38:12] F150W shape = (4377, 9876), valid = 33055333
[21:38:12] F090W shape = (4376, 9875), valid = 33051148


Set DATE-AVG to '2023-11-09T19:55:21.653' from MJD-AVG.
Set DATE-END to '2023-11-09T21:10:19.167' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to    23.148810 from OBSGEO-[XYZ].
Set OBSGEO-H to 1522915295.335 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


## Быстрый поиск центра

Это единственная маленькая функция, которую реально стоит оставить. Без неё код в следующих ячейках станет только грязнее.

In [18]:
def guess_center_fast(img, valid, down=4, sigma=3.0, q=99.5, wcs=None, log=True):
    ny, nx = img.shape

    def _log_center(xc, yc, note=""):
        if not log:
            return
        msg = f"[CENTER-FAST] x={xc:.2f}, y={yc:.2f}"
        if note:
            msg += f" ({note})"
        if wcs is not None:
            ra_deg, dec_deg = wcs.pixel_to_world_values(xc, yc)
            msg += f" | RA={ra_deg:.8f} deg, Dec={dec_deg:.8f} deg"
        print(msg)

    img_d = img[::down, ::down]
    val_d = valid[::down, ::down] & np.isfinite(img_d)

    if not np.any(val_d):
        xc, yc = nx / 2.0, ny / 2.0
        _log_center(xc, yc, note="fallback: no valid downsampled pixels")
        return xc, yc

    data = np.where(val_d, img_d, 0.0).astype(np.float32)
    w = val_d.astype(np.float32)

    num = gaussian_filter(data, sigma=sigma)
    den = gaussian_filter(w, sigma=sigma)
    sm = np.divide(num, den, out=np.full_like(num, np.nan), where=den > 1e-6)

    if not np.isfinite(sm).any():
        xc, yc = nx / 2.0, ny / 2.0
        _log_center(xc, yc, note="fallback: sm is all-NaN")
        return xc, yc

    thr = np.nanpercentile(sm, q)
    sel = np.isfinite(sm) & (sm >= thr)

    if sel.sum() < 50:
        y, x = np.unravel_index(np.nanargmax(sm), sm.shape)
        xc, yc = float(x * down), float(y * down)
        _log_center(xc, yc, note="fallback: argmax")
        return xc, yc

    ys, xs = np.nonzero(sel)
    ws = sm[sel] - np.nanmin(sm[sel])
    ws = np.nan_to_num(ws, nan=0.0) + 1e-12

    x0 = float((xs * ws).sum() / ws.sum()) * down
    y0 = float((ys * ws).sum() / ws.sum()) * down
    _log_center(x0, y0, note=f"down={down}, sigma={sigma}, q={q}")
    return x0, y0

## Подготовка кадра и маски

Здесь либо работаем без фона, либо через двухпроходный фон. Это бывший кусок prepare_image_and_mask, но развёрнуто в явный сценарий.

### finding center

In [19]:
x0_s, y0_s = guess_center_fast(
    img,
    valid150,
    down=4,
    sigma=3.0,
    q=99.5,
    wcs=wcs150,
    log=True,
)
print(f"using center: x0={x0_s:.2f}, y0={y0_s:.2f}")

[03:56:30] [CENTER-FAST] x=6617.25, y=1167.27 (down=4, sigma=3.0, q=99.5) | RA=54.11488622 deg, Dec=-34.97607544 deg
[03:56:30] using center: x0=6617.25, y0=1167.27


### Старая версия маски

In [ ]:
print("preparing image and mask...")

if USE_BKG:
    print("background mode: TWO-PASS")

    # 1) грубый скалярный фон
    sigma_clip = SigmaClip(sigma=3.0, maxiters=10)
    bkg2d_rough = Background2D(
        img_f150,
        box_size=(BKG_BOX0, BKG_BOX0),
        filter_size=(3, 3),
        mask=None,
        sigma_clip=sigma_clip,
        bkg_estimator=MedianBackground(),
        exclude_percentile=10,
    )
    bkg0 = float(np.nanmedian(bkg2d_rough.background))
    img0 = img_f150 - bkg0
    print(f"[BKG] rough scalar background={bkg0:.6g}")

    # 2) маска источников
    ker = Gaussian2DKernel(7.0)
    sm = interpolate_replace_nans(img0, ker)
    bkg_sm, med_sm, rms_sm = sigma_clipped_stats(sm, sigma=2.5, maxiters=5)
    thr = bkg_sm + MASK_NSIGMA * max(rms_sm, 1e-12)

    segm = detect_sources(sm, threshold=thr, npixels=MASK_NPIXELS)
    if segm is None:
        mask_src = np.zeros_like(img0, dtype=bool)
    else:
        if DO_DEBLEND:
            try:
                segm = deblend_sources(sm, segm, npixels=MASK_NPIXELS, nlevels=16, contrast=0.001)
            except Exception as e:
                print(f"[MASK] deblend skipped: {e}")

        labels = segm.labels
        counts = np.bincount(segm.data.ravel(), minlength=labels.max() + 1)
        counts[0] = 0
        big_frac = 0.005
        big_labels = [lab for lab in labels if lab > 0 and counts[lab] > big_frac * img0.size]
        if big_labels:
            segm.remove_labels(big_labels)

        mask_src = segm.make_source_mask(size=5)

    mask = (~valid150) | mask_src

    # 3) финальный 2D-фон уже с маской
    bkg2d = Background2D(
        img_f150,
        box_size=(BKG_BOX2D, BKG_BOX2D),
        filter_size=(3, 3),
        mask=mask,
        sigma_clip=sigma_clip,
        bkg_estimator=MedianBackground(),
        exclude_percentile=10,
    )
    bkg = np.array(bkg2d.background, dtype=float)
    img = img_f150 - bkg

else:
    print("background mode: NO-BKG")
    img = np.array(img_f150, copy=True)

    ker = Gaussian2DKernel(7.0)
    sm = interpolate_replace_nans(img, ker)
    bkg_sm, med_sm, rms_sm = sigma_clipped_stats(sm, sigma=2, maxiters=5)
    thr = bkg_sm + MASK_NSIGMA * max(rms_sm, 1e-12)

    segm = detect_sources(sm, threshold=thr, npixels=MASK_NPIXELS)
    if segm is None:
        mask_src = np.zeros_like(img, dtype=bool)
    else:
        if DO_DEBLEND:
            try:
                segm = deblend_sources(sm, segm, npixels=MASK_NPIXELS, nlevels=16, contrast=0.001)
            except Exception as e:
                print(f"[MASK] deblend skipped: {e}")

        labels = segm.labels
        counts = np.bincount(segm.data.ravel(), minlength=labels.max() + 1)
        counts[0] = 0
        big_frac = 0.005
        big_labels = [lab for lab in labels if lab > 0 and counts[lab] > big_frac * img.size]
        if big_labels:
            segm.remove_labels(big_labels)

        mask_src = segm.make_source_mask(size=5)

    mask = (~valid150) | mask_src
    bkg0 = 0.0
    bkg = np.zeros_like(img, dtype=float)

print(f"mask coverage = {100.0 * mask.sum() / mask.size:.2f}%")

[22:30:34] preparing image and mask...
[22:30:34] background mode: NO-BKG


[22:35:37] [MASK] deblend skipped: No module named 'skimage'
[22:35:38] mask coverage = 25.12%


## Маска 
Сначала убираем яркие штуки сигмой, потом льём Серсика в наны

In [24]:
# --- 1. Ищем яркие компактные объекты до подгонки Серсика ---

# Работаем по исходному кадру без фона и без заливки.
img = np.array(img_f150, copy=True)

# valid150 уже есть: True там, где данные реальны и не NaN/WHT>0.
use_det = valid150 & np.isfinite(img)

# Считаем робастные статистики по реальным пикселям.
# sigma=3.0:
#   берём довольно жёсткий клиппинг, чтобы статистика не слишком
#   искажалась яркими объектами.
# maxiters=5:
#   обычно этого хватает для сходимости, а больше смысла мало.
mean_det, med_det, std_det = sigma_clipped_stats(img[use_det], sigma=3.0, maxiters=5)

# Делаем порог детекции.
# Здесь 3*sigma сверх медианы: идея тут "всё, что больше 3 сигма, это точно не гладкая галактика".
# Используем med_det, а не mean_det, потому что медиана устойчивее к ярким хвостам.
thr_det = med_det + 3.0 * max(std_det, 1e-12)

print(f"[PREMASK] med={med_det:.3e}, std={std_det:.3e}, threshold={thr_det:.3e}")

# detect_sources ищет связные области выше порога.
# npixels=MASK_NPIXELS:
# минимальный размер области, чтобы одиночные шумовые пиксели не считались источником, что кстати может объяснять не вычитание шаровиков. 

segm = detect_sources(img, threshold=thr_det, npixels=MASK_NPIXELS, connectivity=8) # в теории крест в 5 пикселей – эт шаровики

if segm is None:
    premask_src = np.zeros_like(img, dtype=bool)
else:
    if DO_DEBLEND:
        try:
            # nlevels=16:
            #   число уровней разбиения при deblend; это нормальный умеренный дефолт.
            # contrast=0.001:
            #   позволяет разделять достаточно близкие пики, но не уходит в дикий over-deblend.
            segm = deblend_sources(
                img,
                segm,
                npixels=MASK_NPIXELS,
                nlevels=16,
                contrast=0.001,
            )
        except Exception as e:
            print(f"[PREMASK] deblend skipped: {e}")

    # Теперь отбираем только КОМПАКТНЫЕ области.
    # Ты предложил порог 25x25 пикселей.
    # Значит максимальная площадь = 625 пикселей.
    # Всё, что больше, считаем уже не "компактным ярким объектом",
    # а крупной структурой галактики/ядра и не маскируем на этом этапе.
    max_compact_area = 25 * 25

    labels = segm.labels
    counts = np.bincount(segm.data.ravel(), minlength=labels.max() + 1)
    counts[0] = 0

    compact_labels = [lab for lab in labels if 0 < counts[lab] <= max_compact_area]

    premask_src = np.zeros_like(img, dtype=bool)
    if compact_labels:
        for lab in compact_labels:
            premask_src |= (segm.data == lab)

# Финальная предварительная маска для Серсика:
#  - невалидные пиксели
#  - яркие компактные источники
premask = (~valid150) | premask_src

print(f"[PREMASK] coverage = {100.0 * premask.sum() / premask.size:.2f}%")

[04:16:44] [PREMASK] med=2.861e-01, std=3.135e-01, threshold=1.226e+00


[04:29:57] [PREMASK] coverage = 24.22%


### ССерсик

In [25]:
# --- 2. Подгонка гладкой модели Серсика по очищенному кадру ---

yy_full, xx_full = np.indices(img.shape, dtype=float)

# Используем только реальные пиксели, которые не попали в premask.
use_fit = (~premask) & np.isfinite(img)

# Чтобы не закапывать фиттер миллионами точек, прорежаем выборку.
# sample_step=16:
#   это НЕ "правильное физическое число", а чисто вычислительный компромисс.
#   Берём каждую 16-ю точку из уже отфильтрованного массива,
#   чтобы фит не умирал по времени и памяти.
#   Если хочешь более точный, но более медленный фит — уменьшаешь до 8.
#   Если хочешь быстрее — увеличиваешь до 32.
sample_step = 16

y_fit = yy_full[use_fit][::sample_step]
x_fit = xx_full[use_fit][::sample_step]
z_fit = img[use_fit][::sample_step]

print(f"[SERSIC] fit sample size = {z_fit.size}")

# amplitude:
#   стартовую амплитуду берём как 95-й перцентиль по очищенной выборке.
#   Почему не максимум?
#   Потому что максимум легко определяется случайным ярким остатком/артефактом.
#   Почему 95-й?
#   Это просто верхняя яркая часть распределения, но ещё не экстремум.
amp0 = float(np.nanpercentile(z_fit, 95))

# r_eff:
#   стартовый эффективный радиус берём как min(shape)/8.
#   Это не "истина", а стартовый масштаб порядка:
#   если кадр ~4000 пикс по меньшей стороне, то получаем ~500 пикс.
#   Это типичный разумный стартовый масштаб для крупной галактики,
#   чтобы фит не начинал с совсем микроскопического или гигантского размера.
r_eff0 = float(min(img.shape) / 8.0)

sersic_init = Sersic2D(
    # amplitude должен быть >0, поэтому страхуемся от нулей.
    amplitude=max(amp0, 1e-6),

    # r_eff тоже не должен быть нулевым.
    r_eff=max(r_eff0, 10.0),

    # n=4.0:
    #   классический de Vaucouleurs-like старт для ранних типов галактик.
    #   Это просто хороший стартовый guess, а не фиксированная истина.
    n=4.0,

    # x_0, y_0:
    #   центр НЕ грубый, а твой валидный центр, как ты и сказал.
    x_0=float(x0),
    y_0=float(y0),

    # ellip=0.2:
    #   стартовая умеренная эллиптичность.
    #   Это не "мы знаем, что галактика 0.2", а просто безопасный старт,
    #   чтобы модель не начинала ни с круга, ни с экстремально вытянутой формы.
    ellip=0.2,

    # theta=0:
    #   стартовый угол просто нулевой.
    #   Фит потом сам его сдвинет.
    theta=0.0,
)

# Ограничения нужны только для того, чтобы фит не улетел в физический бред.

# amplitude >= 0:
#   отрицательная яркость для гладкой модели галактики бессмысленна.
sersic_init.amplitude.bounds = (0.0, None)

# r_eff от 5 пикс до размера кадра:
#   5 пикс — чтобы не схлопнуться в почти точечный объект;
#   max(img.shape) — чтобы не улететь в безумно большой масштаб.
sersic_init.r_eff.bounds = (5.0, float(max(img.shape)))

# n от 0.5 до 8:
#   ниже 0.5 уже экзотика и обычно фит туда падает по нестабильности;
#   выше 8 часто уже начинается вырожденный бред.
sersic_init.n.bounds = (0.5, 8.0)

# центр должен оставаться внутри кадра.
sersic_init.x_0.bounds = (0.0, float(img.shape[1] - 1))
sersic_init.y_0.bounds = (0.0, float(img.shape[0] - 1))

# ellip от 0 до 0.9:
#   0 — круг;
#   0.9 — уже очень вытянуто, но ещё не математическая катастрофа.
sersic_init.ellip.bounds = (0.0, 0.9)

# theta от -pi/2 до +pi/2:
#   этого диапазона хватает, потому что ориентация эллипса периодична.
sersic_init.theta.bounds = (-np.pi / 2.0, np.pi / 2.0)

fitter = LevMarLSQFitter()

try:
    sersic_fit = fitter(sersic_init, x_fit, y_fit, z_fit)
    print(
        "[SERSIC] fitted:",
        f"amp={float(sersic_fit.amplitude.value):.3e},",
        f"r_eff={float(sersic_fit.r_eff.value):.2f},",
        f"n={float(sersic_fit.n.value):.2f},",
        f"x0={float(sersic_fit.x_0.value):.2f},",
        f"y0={float(sersic_fit.y_0.value):.2f},",
        f"ellip={float(sersic_fit.ellip.value):.3f},",
        f"theta={float(sersic_fit.theta.value):.3f}",
    )
except Exception as e:
    print(f"[SERSIC] fit failed, fallback to initial model: {e}")
    sersic_fit = sersic_init

sersic_model = sersic_fit(xx_full, yy_full)

# --- 3. Заполняем только missing-data области моделью Серсика ---

sm = np.array(img, copy=True)

# bad = там, где данных нет или они невалидны
bad = (~valid150) | (~np.isfinite(sm))

# Только эти области заменяем гладкой моделью
sm[bad] = sersic_model[bad]

[04:34:07] [SERSIC] fit sample size = 2047443


[04:34:16] [SERSIC] fitted: amp=2.435e+00, r_eff=1219.10, n=3.77, x0=6610.24, y0=1166.00, ellip=0.000, theta=1.571


In [30]:
mask_src_path = out_dir / f"{stem}_dbg_mask_src.fits"
mask_path = out_dir / f"{stem}_dbg_mask.fits"
valid_path = out_dir / f"{stem}_dbg_valid150.fits"

fits.writeto(mask_src_path, premask_src.astype(np.uint8), hdr150, overwrite=True)
fits.writeto(mask_path, premask.astype(np.uint8), hdr150, overwrite=True)
fits.writeto(valid_path, valid150.astype(np.uint8), hdr150, overwrite=True)

print(f"[DBG] mask_src → {mask_src_path}")
print(f"[DBG] mask     → {mask_path}")
print(f"[DBG] valid150 → {valid_path}")

[05:05:03] [DBG] mask_src → ../data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_dbg_mask_src.fits
[05:05:03] [DBG] mask     → ../data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_dbg_mask.fits
[05:05:03] [DBG] valid150 → ../data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_dbg_valid150.fits


## Выбор центра галактики

Здесь либо фиксированный центр, либо быстрый автоцентр.

In [10]:
print("choosing galaxy center...")

if FIXED_CENTER is None:
    x0, y0 = guess_center_fast(
        img,
        valid150 & (~mask),
        down=4,
        sigma=3.0,
        q=99.5,
        wcs=wcs150,
        log=True,
    )
    center_src = "auto-fast"
else:
    x0, y0 = FIXED_CENTER
    ra_deg, dec_deg = wcs150.pixel_to_world_values(x0, y0)
    print(f"[CENTER-FIXED] x={x0:.2f}, y={y0:.2f} | RA={ra_deg:.8f} deg, Dec={dec_deg:.8f} deg")
    center_src = "fixed"

print(f"[CENTER] using ({x0:.2f}, {y0:.2f}) [{center_src}]")

[22:49:09] choosing galaxy center...
[22:49:09] [CENTER-FAST] x=6617.25, y=1167.27 (down=4, sigma=3.0, q=99.5) | RA=54.11488624 deg, Dec=-34.97607544 deg
[22:49:09] [CENTER] using (6617.25, 1167.27) [auto-fast]


## Кроп вокруг галактики

Это бывший cutout_box, но без отдельной функции.

In [31]:
print("building cutout...")

ny, nx = sm.shape
x1 = max(0, int(x0 - HALF_SIZE))
x2 = min(nx, int(x0 + HALF_SIZE))
y1 = max(0, int(y0 - HALF_SIZE))
y2 = min(ny, int(y0 + HALF_SIZE))

img_c = sm[y1:y2, x1:x2]
valid_c = valid150[y1:y2, x1:x2]
mask_c = premask_src[y1:y2, x1:x2]

x0c = x0 - x1
y0c = y0 - y1

print(f"[CUTOUT] bounds = x[{x1}:{x2}], y[{y1}:{y2}]")
print(f"[CUTOUT] center in crop = ({x0c:.2f}, {y0c:.2f})")
print(f"[CUTOUT] shape = {img_c.shape}")

[05:06:10] building cutout...
[05:06:10] [CUTOUT] bounds = x[3617:9617], y[0:4167]
[05:06:10] [CUTOUT] center in crop = (3000.25, 1167.27)
[05:06:10] [CUTOUT] shape = (4167, 6000)


## Подготовка данных для изофот

Это вход для Ellipse.

In [32]:
print("preparing data for isophote fit...")

data = img_c.astype(float).copy()

# Не убиваем обратно серсиковскую заливку old-invalid пикселей.
# Маскируем только реальные компактные яркие источники.
data[mask_c] = np.nan

ok = np.isfinite(data)
data_ma = np.ma.array(data, mask=~ok)

print(f"[ISO] finite pixels in crop = {ok.sum()}")

if ok.sum() < 5000:
    raise RuntimeError(f"[ISO] too few valid pixels in crop: N={int(ok.sum())}")
print(f"[ISO] masked by source mask = {mask_c.sum()}, old invalid in crop = {(~valid_c).sum()}")

[05:07:04] preparing data for isophote fit...
[05:07:05] [ISO] finite pixels in crop = 24714054
[05:07:05] [ISO] masked by source mask = 287946, old invalid in crop = 7521786


## Фит изофот

Это главный кусок, который сейчас и надо тестировать отдельно.

In [33]:
print("fitting isophotes...")

geom = EllipseGeometry(
    x0=x0c,
    y0=y0c,
    sma=ISO_START_SMA,
    eps=ISO_START_EPS,
    pa=ISO_START_PA,
)

ell = Ellipse(data_ma, geom)

ny0, nx0 = data.shape
maxsma = 0.90 * float(min(x0c, y0c, (nx0 - 1 - x0c), (ny0 - 1 - y0c)))

# Не даём photutils лезть в sma=0.
# Берём минимальную полуось положительной и заметно больше центральной дырки.
# Если центральная проблемная область у тебя порядка ~8x10 px,
# то minsma=15 px уже безопасно отталкивает фит от нулевой изофоты.
minsma = 15.0

isolist = None
last_err = None

# 1) Основной вариант: линейный шаг + явный minsma
try:
    isolist = ell.fit_image(
        minsma=minsma,
        maxsma=maxsma,
        step=10.0,
        linear=True,
    )
    print("[ISO] fit_image succeeded: linear=True")
except TypeError as e:
    last_err = e
    print(f"[ISO] fit_image(linear=True) signature mismatch: {e}")
except Exception as e:
    last_err = e
    print(f"[ISO] fit_image(linear=True) failed: {e}")

# 2) Упрощённый вариант: без linear, но всё ещё с minsma
if isolist is None:
    try:
        isolist = ell.fit_image(
            minsma=minsma,
            maxsma=maxsma,
            step=10.0,
        )
        print("[ISO] fit_image succeeded: no linear")
    except Exception as e:
        last_err = e
        print(f"[ISO] fit_image(no linear) failed: {e}")

# 3) Совсем консервативный вариант: шаг покрупнее
if isolist is None:
    try:
        isolist = ell.fit_image(
            minsma=minsma,
            maxsma=maxsma,
            step=20.0,
        )
        print("[ISO] fit_image succeeded: coarse step=20")
    except Exception as e:
        last_err = e
        print(f"[ISO] fit_image(coarse) failed: {e}")

if isolist is None or len(isolist) < 10:
    raise RuntimeError(
        f"[ISO] isolist too short: {0 if isolist is None else len(isolist)}; "
        f"last_err={last_err}"
    )

print(f"[ISO] isolist N={len(isolist)}, maxsma≈{float(isolist[-1].sma):.1f} px")

x0_fit = float(np.nanmedian([iso.x0 for iso in isolist]))
y0_fit = float(np.nanmedian([iso.y0 for iso in isolist]))
eps_fit = float(np.nanmedian([iso.eps for iso in isolist]))
pa_fit = float(np.nanmedian([iso.pa for iso in isolist]))
print(f"[ISO] fitted center≈({x0_fit:.1f}, {y0_fit:.1f}) in crop, eps≈{eps_fit:.3f}, pa≈{pa_fit:.3f} rad")

[05:11:26] fitting isophotes...
[05:19:50] [ISO] fit_image succeeded: linear=True
[05:19:50] [ISO] isolist N=104, maxsma≈1050.0 px
[05:19:50] [ISO] fitted center≈(2998.9, 1167.9) in crop, eps≈0.363, pa≈1.899 rad


## Построение модели и residual

Если ты хочешь прямо глазами смотреть, где проблема, это одна из самых важных ячеек.

In [34]:
print("building isophote model...")

model_c = build_ellipse_model(data.shape, isolist)

# Полный кадр модели.
# Размер и форма такие же, как у исходного кадра img.
model = np.full_like(img, np.nan)

# Residual считаем по РЕАЛЬНЫМ данным, а не по sm.
# sm нужен был только для устойчивого фита изофот.
resid = np.array(img, copy=True)

# Вставляем модель в соответствующее место полного кадра.
model[y1:y2, x1:x2] = model_c

# Для residual берём кроп ИЗ ИСХОДНОГО кадра, не из sm.
img_real_c = img[y1:y2, x1:x2]

# Вычитаем модель из реальных данных.
resid[y1:y2, x1:x2] = img_real_c - model_c

# Финальная маска residual:
#    - все старые invalid пиксели,
#    - все яркие компактные источники.
# premask как раз это и содержит.
resid[premask] = np.nan

finite_resid = np.isfinite(resid)
print(
    f"[CHK] resid: finite={finite_resid.sum()}, "
    f"min={np.nanmin(resid):.3e}, med={np.nanmedian(resid):.3e}, max={np.nanmax(resid):.3e}"
)

# Доп. диагностика: сколько валидных пикселей у модели в кропе
finite_model_c = np.isfinite(model_c)
print(
    f"[CHK] model_c: finite={finite_model_c.sum()}, "
    f"min={np.nanmin(model_c):.3e}, med={np.nanmedian(model_c):.3e}, max={np.nanmax(model_c):.3e}"
)

[05:24:40] building isophote model...
[05:24:43] [CHK] resid: finite=32759076, min=-8.022e+01, med=3.053e-01, max=9.172e+02
[05:24:43] [CHK] model_c: finite=25002000, min=0.000e+00, med=0.000e+00, max=3.666e+02


## Карта изофот и overlay

In [35]:
print("building isophote overlay maps...")

# 1. Карта линий изофот в координатах кропа.
nyc, nxc = data.shape
yy, xx = np.indices((nyc, nxc), dtype=float)
iso_map = np.zeros((nyc, nxc), dtype=np.float32)

for iso in isolist:
    x0i = float(getattr(iso, "x0", np.nan))
    y0i = float(getattr(iso, "y0", np.nan))
    sma = float(getattr(iso, "sma", np.nan))
    eps = float(getattr(iso, "eps", np.nan))
    pa  = float(getattr(iso, "pa", np.nan))

    if not (np.isfinite(x0i) and np.isfinite(y0i) and np.isfinite(sma)
            and np.isfinite(eps) and np.isfinite(pa)):
        continue
    if sma <= 0:
        continue

    # q = b/a, а eps = 1 - b/a
    q = max(1e-3, 1.0 - eps)

    cosp = np.cos(pa)
    sinp = np.sin(pa)

    dx = xx - x0i
    dy = yy - y0i

    # Поворот координат в систему эллипса
    xp = dx * cosp + dy * sinp
    yp = -dx * sinp + dy * cosp

    # Эллиптический радиус
    rell = np.sqrt(xp * xp + (yp / q) * (yp / q))

    # Толщина линии изофоты ~1 пиксель
    line = np.abs(rell - sma) <= 0.6
    iso_map[line] = 1.0

# 2. Здесь НЕ используем old-valid mask для зануления, потому что
#    изофоты уже строились на sm с серсиковской заливкой.
#    Если хочешь видеть линии и на залитых областях, оставляем как есть.
#    Но компактные источники можно скрыть из диагностики:
iso_map[mask_c] = np.nan

# 3. Полный кадр карты изофот
iso_full = np.full_like(img, np.nan, dtype=np.float32)
iso_full[y1:y2, x1:x2] = iso_map

# 4. Overlay делаем по ИСХОДНОМУ img, а не по sm.
overlay = np.array(img, dtype=np.float32, copy=True)

good = np.isfinite(overlay)
if np.any(good):
    # 99-й перцентиль нужен как разумный масштаб яркости,
    # чтобы линии были заметны, но не убивали картинку.
    p99 = np.nanpercentile(overlay[good], 99.0)
    amp = float(0.2 * p99) if np.isfinite(p99) else 1.0
else:
    amp = 1.0

lines = np.isfinite(iso_full) & (iso_full > 0.5)
overlay[lines] = np.nan_to_num(overlay[lines], nan=0.0) + amp

if DUMP_ISOPHOTES:
    iso_crop_path = out_dir / f"{stem}_sbf_isophotes_crop.fits"
    iso_full_path = out_dir / f"{stem}_sbf_isophotes_full.fits"
    iso_ovl_path  = out_dir / f"{stem}_sbf_isophotes_overlay.fits"

    fits.writeto(iso_crop_path, iso_map.astype(np.float32), hdr150, overwrite=True)
    fits.writeto(iso_full_path, iso_full.astype(np.float32), hdr150, overwrite=True)
    fits.writeto(iso_ovl_path, overlay.astype(np.float32), hdr150, overwrite=True)

    print(f"[OUT] isophotes(crop)   → {iso_crop_path}")
    print(f"[OUT] isophotes(full)   → {iso_full_path}")
    print(f"[OUT] isophotes(overlay)→ {iso_ovl_path}")

[05:25:08] building isophote overlay maps...
[05:25:44] [OUT] isophotes(crop)   → ../data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_isophotes_crop.fits
[05:25:44] [OUT] isophotes(full)   → ../data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_isophotes_full.fits
[05:25:44] [OUT] isophotes(overlay)→ ../data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_isophotes_overlay.fits


## Сохраняем модель и residual

Это просто удобный артефакт для DS9/SAOImage/FITS Liberator и прочего человеческого страдания.

In [36]:
model_path = out_dir / f"{stem}_sbf_model.fits"
resid_path = out_dir / f"{stem}_sbf_resid.fits"

fits.writeto(model_path, model, hdr150, overwrite=True)
fits.writeto(resid_path, resid, hdr150, overwrite=True)

print(f"[OUT] model → {model_path}")
print(f"[OUT] resid → {resid_path}")

[05:27:00] [OUT] model → ../data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_model.fits
[05:27:00] [OUT] resid → ../data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_sbf_resid.fits


## SBF-аннулус и базовая статистика

Это не финальный ответ статьи, а просто честная проверка, насколько область грязная.

### Получаем геометрию из изофот

In [39]:
print("choosing SBF annulus geometry from isolist...")

# Превращаем isolist в обычные numpy-массивы для удобства
iso_sma = np.array([float(iso.sma) for iso in isolist], dtype=float)
iso_x0  = np.array([float(iso.x0)  for iso in isolist], dtype=float)
iso_y0  = np.array([float(iso.y0)  for iso in isolist], dtype=float)
iso_eps = np.array([float(iso.eps) for iso in isolist], dtype=float)
iso_pa  = np.array([float(iso.pa)  for iso in isolist], dtype=float)

# Выбираем рабочий диапазон полуосей.
# Пока задаём вручную, но уже в координатах ИЗОФОТ, а не круговых радиусов.
#
# Почему 100 и 300:
# - меньше 100 px не лезем, чтобы не трогать ядро и внутренние нестабильные области,
# - до 300 px берём как первый тестовый диапазон, не слишком широкий и не слишком внешний.
#
# Это не "истина", а стартовый разумный диапазон для первой проверки.
SMA_IN_TARGET = 100.0
SMA_OUT_TARGET = 300.0

# Находим ближайшие реальные изофоты к этим полуосям
idx_in = int(np.argmin(np.abs(iso_sma - SMA_IN_TARGET)))
idx_out = int(np.argmin(np.abs(iso_sma - SMA_OUT_TARGET)))

# На всякий случай обеспечиваем правильный порядок
if idx_out <= idx_in:
    raise RuntimeError(
        f"[SBF-ELL] bad annulus indices: idx_in={idx_in}, idx_out={idx_out}"
    )

# Полуоси аннулуса берём по реальным fitted isophotes
sma_in = float(iso_sma[idx_in])
sma_out = float(iso_sma[idx_out])

# Геометрию берём не по одной изофоте, а как медиану по диапазону [idx_in:idx_out]
# Почему медиана:
# - она устойчивее к случайным скачкам fit-а,
# - меньше реагирует на отдельные странные изофоты.
x0_ann = float(np.nanmedian(iso_x0[idx_in:idx_out + 1]))
y0_ann = float(np.nanmedian(iso_y0[idx_in:idx_out + 1]))
eps_ann = float(np.nanmedian(iso_eps[idx_in:idx_out + 1]))
pa_ann = float(np.nanmedian(iso_pa[idx_in:idx_out + 1]))

# q = b/a
q_ann = max(1e-3, 1.0 - eps_ann)

print(
    f"[SBF-ELL] sma_in={sma_in:.1f} px, sma_out={sma_out:.1f} px, "
    f"x0={x0_ann:.2f}, y0={y0_ann:.2f}, eps={eps_ann:.3f}, q={q_ann:.3f}, pa={pa_ann:.3f} rad"
)

[05:43:43] choosing SBF annulus geometry from isolist...
[05:43:43] [SBF-ELL] sma_in=100.0 px, sma_out=300.0 px, x0=2999.23, y0=1168.20, eps=0.292, q=0.708, pa=1.903 rad


### Построение эллиптического аннулуса

In [40]:
print("building elliptical SBF annulus...")

ny_r, nx_r = resid.shape
yy_r, xx_r = np.indices((ny_r, nx_r), dtype=float)

# Сдвиг относительно центра аннулуса
dx = xx_r - x0_ann
dy = yy_r - y0_ann

# Поворот в систему координат эллипса
cosp = np.cos(pa_ann)
sinp = np.sin(pa_ann)

xp = dx * cosp + dy * sinp
yp = -dx * sinp + dy * cosp

# Эллиптический радиус
r_ell = np.sqrt(xp * xp + (yp / q_ann) * (yp / q_ann))

# Эллиптический аннулус между двумя полуосями
annulus_ell = (r_ell >= sma_in) & (r_ell <= sma_out)

# Финальная маска для SBF:
# - premask уже содержит old invalid + яркие компактные источники
# - всё вне эллиптического аннулуса тоже исключаем
mask_sbf = premask | (~annulus_ell)

print(
    f"[SBF-ELL] annulus pixels={int(annulus_ell.sum())}, "
    f"usable pixels={int((~mask_sbf & np.isfinite(resid)).sum())}"
)

[05:43:43] building elliptical SBF annulus...
[05:43:44] [SBF-ELL] annulus pixels=178052, usable pixels=178047


### Проверяем область + базовые статы

In [41]:
print("checking elliptical SBF annulus...")

use = (~mask_sbf) & np.isfinite(resid)
n_use = int(use.sum())

if n_use < 5000:
    raise RuntimeError(f"[SBF-ELL] too few usable pixels in annulus: N={n_use}")

vals = resid[use]

vmin = float(np.nanmin(vals))
vmax = float(np.nanmax(vals))
vmed = float(np.nanmedian(vals))
vmean = float(np.nanmean(vals))
vstd = float(np.nanstd(vals))
vdyn = float(vmax - vmin)

print(
    f"[SBF-REGION-ELL] N={n_use}, "
    f"min={vmin:.3e}, med={vmed:.3e}, max={vmax:.3e}, "
    f"mean={vmean:.3e}, std={vstd:.3e}, dyn={vdyn:.3e}"
)

[05:44:06] checking elliptical SBF annulus...
[05:44:06] [SBF-REGION-ELL] N=178047, min=1.953e-01, med=2.611e-01, max=1.783e+00, mean=2.782e-01, std=6.574e-02, dyn=1.587e+00


## PSF (с кэшем)

Это лучше оставить отдельной ячейкой, потому что PSF — независимая боль.

In [42]:
print("loading/building PSF...")

psf_file = PSFREF if PSFREF is not None else f150w_path
psf_cache_path = out_dir / f"{stem}_psf_{PSF_SIZE}.fits"

if psf_cache_path.exists():
    with fits.open(psf_cache_path, memmap=False) as hd:
        psf = np.array(hd[0].data, dtype=float)
    psf = np.nan_to_num(psf, nan=0.0)
    psf /= psf.sum()
    print(f"[PSF] loaded cache → {psf_cache_path}")
else:
    sim = stpsf.setup_sim_to_match_file(psf_file)
    psf_hdul = sim.calc_psf(nlambda=PSF_NLAMBDA, fov_pixels=PSF_SIZE)

    if isinstance(psf_hdul, fits.HDUList):
        psf = np.array(psf_hdul[0].data, dtype=float)
    elif isinstance(psf_hdul, fits.PrimaryHDU):
        psf = np.array(psf_hdul.data, dtype=float)
    else:
        psf = np.array(psf_hdul, dtype=float)

    if psf.ndim == 3:
        psf = psf.sum(axis=0)

    psf = np.nan_to_num(psf, nan=0.0)
    psf /= psf.sum()

    fits.writeto(psf_cache_path, psf.astype(np.float32), overwrite=True)
    print(f"[PSF] saved cache → {psf_cache_path}")

[05:50:01] loading/building PSF...
[05:50:01] [PSF] loaded cache → ../data/NGC 1380/jw03055-o001_t001_nircam_clear-f150w_i2d_psf_129.fits


##  FFT и P(k), E(k)

Это диагностика формы, не “истина по амплитуде”.

In [53]:
print(f"[FFT] workers={FFT_WORKERS}")

# ------------------------------------------------------------
# Параметры для эмпирического E(k)
# ------------------------------------------------------------
# N_E_REALIZATIONS:
#   сколько шумовых реализаций усреднять для expectation spectrum.
#   16 — разумный компромисс: уже не совсем шумно, но и не слишком долго.
N_E_REALIZATIONS = 16

# KBINS_N:
#   число радиальных бинов по k.
#   80 оставляем как раньше, чтобы сравнение было консистентным.
KBINS_N = 80

# ------------------------------------------------------------
# 1. Рабочее окно измерения
# ------------------------------------------------------------
window = (~mask_sbf) & np.isfinite(resid)

n_use_fft = int(window.sum())
if n_use_fft < 5000:
    raise RuntimeError(f"[FFT] too few usable pixels for FFT: N={n_use_fft}")

W = window.astype(float)

# ------------------------------------------------------------
# 2. P(k) для реального residual
# ------------------------------------------------------------
data_fft = np.zeros_like(resid, dtype=float)
data_fft[window] = resid[window]

# Убираем среднее только в рабочей области
mean_fft = float(np.nanmean(data_fft[window]))
data_fft[window] -= mean_fft

print(f"[FFT] usable pixels={n_use_fft}, mean_removed={mean_fft:.3e}")

with set_workers(FFT_WORKERS):
    F = fft2(data_fft)

# Масштаб P2d пока оставляем простым:
# деление на число используемых пикселей.
P2d = (np.abs(F) ** 2) / float(n_use_fft)

# ------------------------------------------------------------
# 3. Общая k-сетка для 2D -> 1D
# ------------------------------------------------------------
ny, nx = resid.shape

ky = fftshift(fftfreq(ny))
kx = fftshift(fftfreq(nx))
KX, KY = np.meshgrid(kx, ky)
kr = np.hypot(KX, KY)

P2d_s = fftshift(P2d)

kbins = np.linspace(0.0, float(kr.max()), KBINS_N)

Pk_vals = np.full(len(kbins) - 1, np.nan, dtype=float)
Pk_k = np.full_like(Pk_vals, np.nan)

for i in range(len(Pk_vals)):
    sel = (kr >= kbins[i]) & (kr < kbins[i + 1])
    vals = P2d_s[sel]
    if vals.size > 10:
        Pk_vals[i] = float(np.nanmedian(vals))
        Pk_k[i] = 0.5 * (kbins[i] + kbins[i + 1])

mP = np.isfinite(Pk_vals) & np.isfinite(Pk_k) & (Pk_k > 0)
Pk = (Pk_k[mP], Pk_vals[mP])

print(f"[FFT] P(k) bins kept = {mP.sum()}")

# ------------------------------------------------------------
# 4. Подготовка большого PSF-ядра для свёртки шума
# ------------------------------------------------------------
big_psf = np.zeros((ny, nx), dtype=float)

py, px = psf.shape
y0_psf = ny // 2 - py // 2
x0_psf = nx // 2 - px // 2
big_psf[y0_psf:y0_psf + py, x0_psf:x0_psf + px] = psf

# ------------------------------------------------------------
# 5. Эмпирический E(k):
#    белый шум -> свёртка с PSF -> то же окно -> то же вычитание среднего -> FFT
# ------------------------------------------------------------
Ek_stack = []

rng = np.random.default_rng(None)

for i_real in range(N_E_REALIZATIONS):
    # Белый шум N(0,1) на полном кадре
    noise = rng.normal(loc=0.0, scale=1.0, size=(ny, nx))

    # Свёртка белого шума с PSF через FFT
    with set_workers(FFT_WORKERS):
        F_noise = fft2(noise)
        F_psf = fft2(big_psf)
        sim = np.real(np.fft.ifft2(F_noise * F_psf))

    # Применяем ТО ЖЕ окно, что и для данных
    sim_masked = np.zeros_like(sim, dtype=float)
    sim_masked[window] = sim[window]

    # Вычитаем среднее по той же рабочей области
    sim_mean = float(np.nanmean(sim_masked[window]))
    sim_masked[window] -= sim_mean

    # FFT симулированного поля
    with set_workers(FFT_WORKERS):
        F_sim = fft2(sim_masked)

    E2d_sim = (np.abs(F_sim) ** 2) / float(n_use_fft)
    E2d_sim_s = fftshift(E2d_sim)

    Ek_vals_i = np.full(len(kbins) - 1, np.nan, dtype=float)

    for j in range(len(Ek_vals_i)):
        sel = (kr >= kbins[j]) & (kr < kbins[j + 1])
        vals = E2d_sim_s[sel]
        if vals.size > 10:
            Ek_vals_i[j] = float(np.nanmedian(vals))

    Ek_stack.append(Ek_vals_i)

Ek_stack = np.array(Ek_stack, dtype=float)

# Усредняем expectation spectrum по реализациям
Ek_vals = np.nanmedian(Ek_stack, axis=0)
Ek_k = 0.5 * (kbins[:-1] + kbins[1:])

mE = np.isfinite(Ek_vals) & np.isfinite(Ek_k) & (Ek_k > 0)
Ek = (Ek_k[mE], Ek_vals[mE])

print(f"[FFT] E(k) bins kept = {mE.sum()}")
print(f"[FFT] E(k) realizations = {N_E_REALIZATIONS}")
print("[FFT] P(k) and empirical E(k) ready")

[06:29:00] [FFT] workers=-1
[06:29:01] [FFT] usable pixels=178047, mean_removed=2.782e-01
[06:29:06] [FFT] P(k) bins kept = 79
[06:31:37] [FFT] E(k) bins kept = 79
[06:31:37] [FFT] E(k) realizations = 16
[06:31:37] [FFT] P(k) and empirical E(k) ready


## Фит P0

In [54]:
print("fitting P(k) = P0 * E(k) + P1 ...")

# ------------------------------------------------------------------
# 1. Рабочее окно по пространственным частотам
# ------------------------------------------------------------------
# kmin=0.03:
#   отрезаем самые низкие частоты, где обычно сидят ошибки гладкой модели,
#   остаточные градиенты и оконная функция аннулуса.
#
# kmax=0.40:
#   отрезаем самые высокие частоты, где уже доминируют шум, дискретизация
#   и прочие мелкие гадости.
#
# Это пока эвристический диапазон, но для первой проверки он разумный.
kmin, kmax = 0.03, 0.40

kP, P = Pk
kE, E = Ek

sel = (kP >= kmin) & (kP <= kmax)

if int(sel.sum()) < 10:
    raise RuntimeError(f"[FIT] too few P(k) points in k-window: N={int(sel.sum())}")

# Интерполируем E(k) на сетку P(k), чтобы сравнивать одно с другим по одним и тем же k.
E_int = np.interp(kP[sel], kE, E, left=np.nan, right=np.nan)

x = E_int
y = P[sel]

# Оставляем только конечные и положительные значения.
# P(k) и E(k) физически должны быть > 0.
m = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
x = x[m]
y = y[m]

if x.size < 10:
    raise RuntimeError(f"[FIT] too few valid points after cleaning: N={x.size}")

# ------------------------------------------------------------------
# 2. Диагностика: насколько форма P(k) вообще похожа на E(k)
# ------------------------------------------------------------------
corr = float(np.corrcoef(x, y)[0, 1])
print(f"[FIT] window k={kmin:.3f}..{kmax:.3f}, N={x.size}, corr(E,P)≈{corr:.3f}")

# Если корреляция совсем слабая, P0 интерпретировать будет опасно.
# Но пока не падаем, а просто честно печатаем предупреждение.
if not np.isfinite(corr) or abs(corr) < 0.3:
    print("[FIT] WARNING: weak correlation between E(k) and P(k)")

# ------------------------------------------------------------------
# 3. Линейный фит y = P0 * x + P1
# ------------------------------------------------------------------
A = np.vstack([x, np.ones_like(x)]).T
P0, P1 = np.linalg.lstsq(A, y, rcond=None)[0]

# frac = какая доля модели приходится на PSF-подобную компоненту
den = P0 + P1
frac = float(P0 / den) if np.isfinite(den) and den != 0 else np.nan

print(f"[FIT] P0={P0:.3e}, P1={P1:.3e}, frac={frac:.3f}")

# ------------------------------------------------------------------
# 4. Доп. sanity-check
# ------------------------------------------------------------------
if not np.isfinite(P0) or not np.isfinite(P1):
    print("[FIT] WARNING: non-finite fit coefficients")

if P0 <= 0:
    print("[FIT] WARNING: P0 <= 0, physical SBF interpretation is doubtful")

if np.isfinite(P1) and abs(P1) > 5.0 * max(P0, 1e-30):
    print("[FIT] WARNING: |P1| >> P0, white-noise term dominates fit")

[06:33:41] fitting P(k) = P0 * E(k) + P1 ...
[06:33:41] [FIT] window k=0.030..0.400, N=42, corr(E,P)≈0.904
[06:33:41] [FIT] P0=2.036e-01, P1=5.923e-03, frac=0.972


## Базовый расчёт m̄

Это твой текущий способ через var_sbf / Imean.

In [ ]:
vals_sbf = resid[use]
lo, hi = np.nanpercentile(vals_sbf, [5, 95])
core_mask = (vals_sbf >= lo) & (vals_sbf <= hi)

var_sbf = float(np.nanvar(vals_sbf[core_mask]))
Imean = float(np.nanmean(model[(~mask_sbf) & np.isfinite(model)]))
Pf = var_sbf / Imean

jy_per_pix = 2.350443e-5 * pix_area
zp_ab = float(-2.5 * np.log10(jy_per_pix / 3631.0))
mbar = -2.5 * np.log10(Pf) + zp_ab

print(f"[SBF-DBG] P0={P0:.3e}, var_sbf={var_sbf:.3e}, ratio={var_sbf / P0:.3e}")
print(f"[SBF] sigma^2={var_sbf:.3e}, Imean={Imean:.3e}, Pf={Pf:.3e}")
print(f"m̄(F150W) = {mbar:.3f} mag")

## Цвет по F090W/F150W

Грубая диагностика, как и у тебя было.

In [ ]:
if img_f090 is not None:
    use_color = (~mask) & valid150 & valid090 & np.isfinite(img) & np.isfinite(img_f090)
    if use_color.sum() > 100:
        f150_med = float(np.nanmedian(img[use_color]))
        f090_med = float(np.nanmedian(img_f090[use_color]))
        if f150_med > 0 and f090_med > 0:
            color = -2.5 * np.log10(f090_med / f150_med)
            print(f"(F090W - F150W) ≈ {color:.3f} mag")

## circles scan

In [ ]:
print("scan SBF annuli...")

rows = []

yy_r, xx_r = np.ogrid[:resid.shape[0], :resid.shape[1]]
rr_r = np.hypot(yy_r - y0_sbf, xx_r - x0_sbf)

r_max_img = min(
    np.hypot(y0_sbf, x0_sbf),
    np.hypot(y0_sbf, resid.shape[1] - 1 - x0_sbf),
    np.hypot(resid.shape[0] - 1 - y0_sbf, x0_sbf),
    np.hypot(resid.shape[0] - 1 - y0_sbf, resid.shape[1] - 1 - x0_sbf),
)

if r_max_img > 800.0:
    r_min, r_max, step, width = 100.0, min(800.0, r_max_img - 50.0), 50.0, 100.0
else:
    r_min, r_max, step, width = 30.0, max(60.0, r_max_img - 20.0), 20.0, 60.0

print(f"[SCAN] scan radii: rin∈[{r_min:.0f},{r_max:.0f}], width≈{width:.0f} px")

for rin in np.arange(r_min, r_max, step):
    rout = rin + width
    if rout > r_max_img:
        rout = r_max_img

    annulus = (rr_r >= rin) & (rr_r <= rout)
    mask_scan = mask | (~annulus)

    use_scan = (~mask_scan) & np.isfinite(resid)
    n = int(use_scan.sum())
    if n < 5000:
        continue

    vals = resid[use_scan]
    lo, hi = np.nanpercentile(vals, [5, 95])
    core_mask = (vals >= lo) & (vals <= hi)
    if core_mask.sum() < 100:
        continue

    var_scan = float(np.nanvar(vals[core_mask]))
    use_I = (~mask_scan) & np.isfinite(model)
    if use_I.sum() < 5000:
        continue

    Imean_scan = float(np.nanmean(model[use_I]))
    if not np.isfinite(Imean_scan) or Imean_scan <= 0:
        continue

    Pf_scan = var_scan / Imean_scan
    if not np.isfinite(Pf_scan) or Pf_scan <= 0:
        continue

    mbar_scan = -2.5 * np.log10(Pf_scan) + zp_ab
    dyn_scan = float(np.nanmax(vals) - np.nanmin(vals))
    std_scan = float(np.nanstd(vals))

    print(
        f"[SCAN] rin={rin:5.1f}, rout={rout:5.1f}, "
        f"N={n:7d}, Imean={Imean_scan:.3e}, std={std_scan:.3e}, "
        f"dyn={dyn_scan:.3e}, var_sbf={var_scan:.3e}, "
        f"Pf={Pf_scan:.3e}, m̄={mbar_scan:.3f}"
    )

    rows.append({
        "rin": float(rin),
        "rout": float(rout),
        "mbar": float(mbar_scan),
    })

## Поиск плато

Последняя диагностическая ячейка.

In [ ]:
print("searching plateau...")

if rows:
    rows_sorted = sorted(rows, key=lambda r: r["rin"])
    best = None
    mvals = [r["mbar"] for r in rows_sorted]
    dm_max = 0.4
    min_len = 3

    n = len(rows_sorted)
    for i in range(n):
        cur_min = mvals[i]
        cur_max = mvals[i]
        for j in range(i + 1, n):
            v = mvals[j]
            cur_min = min(cur_min, v)
            cur_max = max(cur_max, v)
            if cur_max - cur_min > dm_max:
                break
            length = j - i + 1
            if length >= min_len:
                rin = rows_sorted[i]["rin"]
                rout = rows_sorted[j]["rout"]
                rmid = 0.5 * (rin + rout)
                score = (length, -abs(rmid))
                if (best is None) or (score > best[0]):
                    best = (score, i, j)

    if best is None:
        print("[PLATEAU] not found")
    else:
        _, i0, j0 = best
        subset = rows_sorted[i0:j0+1]
        rin_min = subset[0]["rin"]
        rout_max = subset[-1]["rout"]
        m_list = [r["mbar"] for r in subset]
        print(
            f"[PLATEAU] rin≈{rin_min:.1f}..{rout_max:.1f} px, "
            f"m̄≈{np.mean(m_list):.3f}, Δm̄={max(m_list)-min(m_list):.3f}, "
            f"N_rings={len(subset)}"
        )
else:
    print("[PLATEAU] no rows")